In [1]:
# ----------------------------
# Bank System — Data Driven Demo
# ----------------------------

import sys
sys.path.append("../src")

from datetime import datetime, date
from collections import defaultdict

from bank.bank import Bank
from bank.client import Client, Contact

from account.bank_account import BankAccount
from account.savings_account import SavingsAccount
from account.premium_account import PremiumAccount

from account.enums import AccountCurrency

from transaction.transaction import Transaction
from transaction.transaction_queue import TransactionAlreadyInQueue, QueueIsEmpty


# ----------------------------
# RAW DATA (можно вынести в JSON)
# ----------------------------

DATA = {
    "clients": [
        {"id": "C001", "name": "Ivan Petrov"},
        {"id": "C002", "name": "Anna Smirnova"},
        {"id": "C003", "name": "John Smith"},
        {"id": "C004", "name": "Maria Garcia"},
        {"id": "C005", "name": "Liam Brown"},
        {"id": "C006", "name": "Emma Wilson"},
        {"id": "C007", "name": "Noah Martinez"},
    ],

    "accounts": [
        {"id": "A001", "client_id": "C001", "type": "debit", "balance": 5000, "currency": "USD"},
        {"id": "A002", "client_id": "C001", "type": "savings", "balance": 2000, "currency": "EUR"},
        {"id": "A003", "client_id": "C002", "type": "debit", "balance": 8000, "currency": "USD"},
        {"id": "A004", "client_id": "C002", "type": "debit", "balance": 1500, "currency": "EUR"},
        {"id": "A005", "client_id": "C003", "type": "premium", "balance": 12000, "currency": "USD"},
        {"id": "A006", "client_id": "C004", "type": "debit", "balance": 3000, "currency": "EUR"},
        {"id": "A007", "client_id": "C004", "type": "savings", "balance": 700, "currency": "USD"},
        {"id": "A008", "client_id": "C005", "type": "debit", "balance": 4000, "currency": "USD"},
        {"id": "A009", "client_id": "C006", "type": "premium", "balance": 9500, "currency": "EUR"},
        {"id": "A010", "client_id": "C007", "type": "debit", "balance": 1000, "currency": "USD"},
    ],

    "transactions": [
        ("A001","A003",500,"USD"),
        ("A002","A006",300,"EUR"),
        ("A003","A005",7000,"USD"),
        ("A004","A010",2000,"EUR"),  # insufficient
        ("A005","A001",10000,"USD"), # large
        ("A007","A008",600,"USD"),
        ("A008","A009",3500,"USD"),  # cross
        ("A009","A002",1000,"EUR"),
        ("A010","A003",200,"USD"),
        ("A001","A003",99999,"USD"), # limit exceeded
        ("A003","A004",9000,"USD"),  # suspicious
        ("A006","A007",500,"EUR"),
        ("A002","A003",1500,"EUR"),
        ("A007","A001",100,"USD"),
        ("A005","A009",11000,"USD"), # very large
        ("A009","A003",8000,"EUR"),  # suspicious
    ]
}


# ----------------------------
# Create bank
# ----------------------------

bank = Bank()
print("Bank created")


# ----------------------------
# Create clients
# ----------------------------

client_map = {}

for c in DATA["clients"]:
    client = Client(
        name=c["name"],
        client_id=c["id"],
        date_of_birth=date(1990, 1, 1),
        login=c["id"],
        password="test",
        contacts=[Contact("email", f"{c['id']}@test.com")]
    )
    bank.add_client(client)
    client_map[c["id"]] = client

print(f"Clients loaded: {len(client_map)}")


# ----------------------------
# Create accounts
# ----------------------------

account_map = {}

def create_account(acc_data, owner_name):
    if acc_data["type"] == "debit":
        return BankAccount(owner_name=owner_name)
    elif acc_data["type"] == "savings":
        return SavingsAccount(owner_name=owner_name, min_balance=100, monthly_interest=1)
    elif acc_data["type"] == "premium":
        return PremiumAccount(owner_name=owner_name, fee=5)


for acc in DATA["accounts"]:
    client = client_map[acc["client_id"]]

    account = create_account(acc, client.name)
    bank.open_account(client.client_id, account)

    # стартовый баланс
    account.deposit(acc["balance"])

    account_map[acc["id"]] = account

print(f"Accounts loaded: {len(account_map)}")


# ----------------------------
# Create transactions
# ----------------------------

queue = bank.transaction_queue
transactions = []

for from_id, to_id, amount, currency in DATA["transactions"]:

    # invalid account simulation
    to_account = account_map.get(to_id)

    if not to_account:
        continue

    t = Transaction(
        amount,
        AccountCurrency[currency],
        1,
        account_map[from_id],
        to_account
    )

    transactions.append(t)

    try:
        queue.enqueue(t)
    except TransactionAlreadyInQueue:
        pass


# ----------------------------
# Add suspicious patterns
# ----------------------------

# frequent transactions
for _ in range(6):
    queue.enqueue(
        Transaction(
            200,
            AccountCurrency.USD,
            1,
            account_map["A001"],
            account_map["A003"]
        )
    )

# freeze scenario
bank.freeze_account(account_map["A010"].id)

queue.enqueue(
    Transaction(
        300,
        AccountCurrency.USD,
        1,
        account_map["A001"],
        account_map["A010"]
    )
)

print(f"Transactions queued: {len(transactions)}")


# ----------------------------
# Process queue
# ----------------------------

processor = bank.transaction_processor

print("\nProcessing queue...")

while True:
    try:
        processor.process()
    except QueueIsEmpty:
        print("Queue empty")
        break


# ----------------------------
# Results
# ----------------------------

print("\nTransaction results:")

for t in transactions:
    print(
        t.transaction_id,
        t.status.name,
        "| reject:", t.reject_reason
    )


# ----------------------------
# Suspicious report
# ----------------------------

print("\nSuspicious transactions:")

for log in bank.audit_log.get_suspicious_transactions():
    print(
        getattr(log, "client_id", "-"),
        getattr(log, "risk", "-"),
    )


# ----------------------------
# Risk profiles
# ----------------------------

print("\nRisk profiles:")

for client_id in client_map:
    profile = bank.audit_log.get_client_risk_profile(client_id)
    print(client_id, dict(profile))


# ----------------------------
# Errors
# ----------------------------

print("\nErrors:")

for err in bank.audit_log.get_errors():
    print(err)


# ----------------------------
# Final balances
# ----------------------------

print("\nFinal balances:")

for acc_id, acc in account_map.items():
    print(acc_id, acc.balance)

Bank created
Clients loaded: 7
Accounts loaded: 10
Transactions queued: 16

Processing queue...
Queue empty

Transaction results:
8df37c80-11af-4eeb-981a-9dbf8f954c9c EXECUTED | reject: None
8c6a18cb-dfd4-4e15-972b-fd28ec4cd980 EXECUTED | reject: None
ba0aadde-fc79-4106-b67e-9870479902f2 CANCELED | reject: None
443a0b01-feed-4140-956e-96b700f0456c REJECTED | reject: 
d994241e-e42f-404c-81ac-e28c329d4f17 CANCELED | reject: None
869e8b04-c445-4f0f-9f24-d53b1cfa5e9d EXECUTED | reject: None
47d3decc-e0e4-4a6f-958b-e6cccf4a2dad CANCELED | reject: None
27c5e2b2-b9cc-455b-ad2d-c6d8d80a356a EXECUTED | reject: None
4bd506cf-45c4-4381-aa73-fb92947d097c REJECTED | reject: 
2fb8fede-fe4a-442c-a9e0-e252c43150ae CANCELED | reject: None
e56fb658-301b-4d80-bbb4-1ad025247ac1 CANCELED | reject: None
c02a9a69-5a73-4ca9-8931-fa2b738a9d23 EXECUTED | reject: None
a81b4fdf-ea68-489a-8aa4-9e557744d976 EXECUTED | reject: None
56114834-b093-4040-8721-d4a50b162a73 EXECUTED | reject: None
4c60b980-9e55-4dcb-8a13-

TypeError: 'NoneType' object is not iterable